In [1]:
# =====================================================================
# CÉLULA 1 — Importações
# =====================================================================
import hashlib
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

ENCODING_AFD = "iso-8859-1"   # Anexo V, Portaria MTP 671/2021, item 2

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

In [2]:
# =====================================================================
# CÉLULA 2 — Caminhos dinâmicos
# Funciona em notebook, script .py e executável gerado pelo PyInstaller
# =====================================================================
def get_base_dir() -> Path:
    """Diretório base do programa, independente de onde ele está rodando."""
    # 1) Executável do PyInstaller
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent

    # 2) Notebook no VSCode
    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        return Path(vsc).resolve().parent

    # 3) Script .py comum
    arquivo = globals().get("__file__")
    if arquivo:
        return Path(arquivo).resolve().parent

    # 4) Jupyter/Colab
    return Path.cwd()


BASE_DIR = get_base_dir()


def resolver_pasta_arquivos(nome: str = "arquivos") -> Path:
    """
    Pasta onde ficam os AFD.
    Ordem de prioridade:
      1. Variável de ambiente AFD_PASTA
      2. <pasta do executável>/arquivos
      3. <_MEIPASS>/arquivos  (PyInstaller --onefile, se empacotado como dado)
      4. cria a pasta ao lado do executável
    """
    env = os.getenv("AFD_PASTA")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]

    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_dir():
            return candidato

    destino = BASE_DIR / nome
    destino.mkdir(parents=True, exist_ok=True)
    return destino


PASTA_ARQUIVOS = resolver_pasta_arquivos()

print("BASE_DIR       :", BASE_DIR)
print("PASTA_ARQUIVOS :", PASTA_ARQUIVOS)
print("Existe         :", PASTA_ARQUIVOS.is_dir())

BASE_DIR       : \\AGROSRV\Automacoes\Marcacao_Pontos
PASTA_ARQUIVOS : \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
Existe         : True


In [3]:
# =====================================================================
# CÉLULA 3 — Arquivos AFD encontrados
# =====================================================================
EXTENSOES_AFD = {"", ".txt", ".afd", ".dat"}


def parece_afd(caminho: Path) -> bool:
    """Confere se a 1ª linha tem o formato de cabeçalho do AFD."""
    try:
        with caminho.open("r", encoding=ENCODING_AFD, errors="replace") as f:
            primeira = f.readline().strip()
    except OSError:
        return False
    return len(primeira) >= 10 and primeira[:9].isdigit() and primeira[9] == "1"


def listar_arquivos_afd(pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or PASTA_ARQUIVOS)
    if not pasta.is_dir():
        return []
    return [
        item for item in sorted(pasta.iterdir())
        if item.is_file()
        and item.suffix.lower() in EXTENSOES_AFD
        and parece_afd(item)
    ]


ARQUIVOS = listar_arquivos_afd()
print(f"{len(ARQUIVOS)} arquivo(s) AFD em {PASTA_ARQUIVOS}")
for arq in ARQUIVOS:
    print(f"  - {arq.name}  ({arq.stat().st_size / 1024:.1f} KB)")

1 arquivo(s) AFD em \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
  - AFDBR512021002936436585411000133REP_P.txt  (113.7 KB)


In [4]:
# =====================================================================
# CÉLULA 4 — Leiaute oficial (posições 1-based, Anexo V)
# =====================================================================
CAMPOS: dict[str, dict[str, tuple[int, int]]] = {
    "1": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "tipo_id_empregador": (11, 1),
        "cnpj_cpf_empregador": (12, 14),
        "cno_caepf": (26, 14),
        "razao_social": (40, 150),
        "num_fabricacao_inpi": (190, 17),
        "data_inicial": (207, 10),
        "data_final": (217, 10),
        "dh_geracao": (227, 24),
        "versao": (251, 3),
        "tipo_id_fabricante": (254, 1),
        "cnpj_cpf_fabricante": (255, 14),
        "modelo": (269, 30),
        "crc": (299, 4),
    },
    "2": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "cpf_responsavel": (35, 14),
        "tipo_id_empregador": (49, 1),
        "cnpj_cpf_empregador": (50, 14),
        "cno_caepf": (64, 14),
        "razao_social": (78, 150),
        "local_prestacao": (228, 100),
        "crc": (328, 4),
    },
    "3": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_marcacao": (11, 24),
        "cpf": (35, 12),
        "crc": (47, 4),
    },
    "4": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_antes_ajuste": (11, 24),
        "dh_ajustada": (35, 24),
        "cpf_responsavel": (59, 11),
        "crc": (70, 4),
    },
    "5": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "operacao": (35, 1),
        "cpf": (36, 12),
        "nome": (48, 52),
        "dados_identificacao": (100, 4),
        "cpf_responsavel": (104, 11),
        "crc": (115, 4),
    },
    "6": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_gravacao": (11, 24),
        "evento": (35, 2),
    },
    "7": {
        "nsr": (1, 9),
        "tipo": (10, 1),
        "dh_marcacao": (11, 24),
        "cpf": (35, 12),
        "dh_gravacao": (47, 24),
        "coletor": (71, 2),
        "online_offline": (73, 1),
        "hash": (74, 64),
    },
}

TAMANHO_REGISTRO = {"1": 302, "2": 331, "3": 50,
                    "4": 73, "5": 118, "7": 137, "9": 64}

COLETOR = {
    "01": "aplicativo mobile",
    "02": "browser (navegador)",
    "03": "aplicativo desktop",
    "04": "dispositivo eletrônico",
    "05": "outro dispositivo eletrônico",
}

EVENTO_REP = {
    "01": "abertura do REP por manutenção/violação",
    "02": "retorno de energia",
    "03": "introdução de dispositivo externo (Porta Fiscal)",
    "04": "retirada de dispositivo externo (Porta Fiscal)",
    "05": "emissão da Relação Instantânea de Marcações",
    "06": "erro de impressão",
    "07": "disponibilidade de serviço",
    "08": "indisponibilidade de serviço",
}

OPERACAO = {"I": "inclusão", "A": "alteração", "E": "exclusão"}

QTD_TRAILER = {"2": (10, 9), "3": (19, 9), "4": (28, 9),
               "5": (37, 9), "6": (46, 9), "7": (55, 9)}

ASSINATURA_P7S = "ASSINATURA_DIGITAL_EM_ARQUIVO_P7S"
TIPOS_COM_CRC = ("1", "2", "3", "4", "5")

In [5]:
# =====================================================================
# CÉLULA 5 — Utilidades de campo, datas e CRC-16
# =====================================================================
def extrair(linha: str, posicao: int, tamanho: int) -> str:
    """Extrai campo usando posição inicial 1-based, como no Anexo V."""
    inicio = posicao - 1
    return linha[inicio:inicio + tamanho]


def limpar(valor: str) -> str:
    return (valor or "").strip()


def somente_digitos(valor: str) -> str:
    return re.sub(r"\D", "", valor or "")


def formatar_cpf(cpf: str) -> str:
    # O campo tem 12 posições e o CPF (11 dígitos) vem alinhado à direita:
    # "009281928817" -> 928.192.881-7 é o CPF; o zero é preenchimento.
    d = somente_digitos(cpf)[-11:]
    return f"{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}" if len(d) == 11 else limpar(cpf)


def formatar_cnpj(cnpj: str) -> str:
    d = somente_digitos(cnpj)
    if len(d) == 14:
        return f"{d[:2]}.{d[2:5]}.{d[5:8]}/{d[8:12]}-{d[12:]}"
    return limpar(cnpj)


def parse_data_hora(valor: str) -> datetime | None:
    """Campo DH do Anexo V: 2021-04-27T16:44:00-0300"""
    texto = limpar(valor)
    if not texto:
        return None
    for fmt in ("%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(texto, fmt)
        except ValueError:
            continue
    try:
        return pd.to_datetime(texto, format="ISO8601").to_pydatetime()
    except Exception:
        return None


def parse_data(valor: str):
    texto = limpar(valor)
    if not texto:
        return None
    try:
        return datetime.strptime(texto, "%Y-%m-%d").date()
    except ValueError:
        return None


def crc16_kermit(dados: bytes, crc: int = 0x0000) -> int:
    """
    CRC-16/CCITT-TRUE (KERMIT), exigido pela Portaria 671 para AFD de REP-A e REP-P.
    Vetor de teste: b'123456789' -> 0x2189
    """
    for byte in dados:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0x8408 if crc & 0x0001 else crc >> 1
    return crc & 0xFFFF


def calcular_crc(linha: str, tamanho_crc: int = 4) -> str:
    return f"{crc16_kermit(linha[:-tamanho_crc].encode(ENCODING_AFD, errors='replace')):04X}"


def crc_valido(linha: str, tamanho_crc: int = 4) -> bool:
    if len(linha) <= tamanho_crc:
        return False
    return calcular_crc(linha, tamanho_crc) == limpar(linha[-tamanho_crc:]).upper()

In [6]:
# =====================================================================
# CÉLULA 6 — Parser tipo "1" (cabeçalho)
# =====================================================================
def parse_tipo_1(linha: str) -> dict:
    c = CAMPOS["1"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    id_emp = campo("tipo_id_empregador")
    documento = somente_digitos(campo("cnpj_cpf_empregador"))

    return {
        "tipo": "1",
        "nsr": campo("nsr"),
        "tipo_id_empregador": id_emp,
        "empregador_documento": formatar_cnpj(documento) if id_emp == "1" else formatar_cpf(documento),
        "empregador_razao_social": campo("razao_social"),
        "cno_caepf": campo("cno_caepf"),
        "numero_fabricacao_inpi": campo("num_fabricacao_inpi"),
        "data_inicial": parse_data(campo("data_inicial")),
        "data_final": parse_data(campo("data_final")),
        "dh_geracao": parse_data_hora(campo("dh_geracao")),
        "versao_leiaute": campo("versao"),
        "fabricante_documento": campo("cnpj_cpf_fabricante"),
        "modelo_rep": campo("modelo"),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha) if len(linha) >= 302 else None,
    }

In [7]:
# =====================================================================
# CÉLULA 7 — Parser tipo "2" (inclusão/alteração da empresa no REP)
# =====================================================================
def parse_tipo_2(linha: str) -> dict:
    c = CAMPOS["2"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    return {
        "tipo": "2",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "tipo_id_empregador": campo("tipo_id_empregador"),
        "empregador_documento": campo("cnpj_cpf_empregador"),
        "cno_caepf": campo("cno_caepf"),
        "empregador_razao_social": campo("razao_social"),
        "local_prestacao": campo("local_prestacao"),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [8]:
# =====================================================================
# CÉLULA 8 — Parser tipo "3" (marcação de ponto REP-C e REP-A)
# =====================================================================
def parse_tipo_3(linha: str) -> dict:
    c = CAMPOS["3"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))
    dh_raw = campo("dh_marcacao")

    return {
        "tipo": "3",
        "nsr": campo("nsr"),
        "dh_marcacao_raw": dh_raw,
        "dh_marcacao": parse_data_hora(dh_raw),
        "cpf": somente_digitos(campo("cpf")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [9]:
# =====================================================================
# CÉLULA 9 — Parser tipo "4" (ajuste do relógio)
# =====================================================================
def parse_tipo_4(linha: str) -> dict:
    c = CAMPOS["4"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    antes = parse_data_hora(campo("dh_antes_ajuste"))
    depois = parse_data_hora(campo("dh_ajustada"))
    delta = (depois - antes).total_seconds() if antes and depois else None

    return {
        "tipo": "4",
        "nsr": campo("nsr"),
        "dh_antes_ajuste": antes,
        "dh_ajustada": depois,
        "delta_segundos": delta,
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [10]:
# =====================================================================
# CÉLULA 10 — Parser tipo "5" (inclusão/alteração/exclusão de empregado)
# =====================================================================
def parse_tipo_5(linha: str) -> dict:
    c = CAMPOS["5"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))
    operacao = campo("operacao").upper()

    return {
        "tipo": "5",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "operacao": operacao,
        "operacao_desc": OPERACAO.get(operacao, operacao),
        "cpf": somente_digitos(campo("cpf")),
        "nome": campo("nome"),
        "dados_identificacao": campo("dados_identificacao"),
        "cpf_responsavel": formatar_cpf(campo("cpf_responsavel")),
        "crc": campo("crc"),
        "crc_ok": crc_valido(linha),
    }

In [11]:
# =====================================================================
# CÉLULA 11 — Parser tipo "6" (eventos sensíveis do REP)
# Observação: o Anexo V não fixa CRC-16 para o tipo 6; o resíduo é
# preservado em "complemento" para conferência manual.
# =====================================================================
def parse_tipo_6(linha: str) -> dict:
    c = CAMPOS["6"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    evento = campo("evento")
    resto = linha[36:]                      # após o campo 4 (posições 35-36)
    complemento = resto
    crc = ""
    if re.fullmatch(r"[0-9A-Fa-f]{4}", resto[-4:] or ""):
        complemento, crc = resto[:-4], resto[-4:]

    return {
        "tipo": "6",
        "nsr": campo("nsr"),
        "dh_gravacao": parse_data_hora(campo("dh_gravacao")),
        "evento": evento,
        "evento_desc": EVENTO_REP.get(evento, evento),
        "complemento": limpar(complemento),
        "crc": crc,
    }

In [12]:
# =====================================================================
# CÉLULA 12 — Parser tipo "7" (marcação de ponto REP-P)
# =====================================================================
def parse_tipo_7(linha: str) -> dict:
    c = CAMPOS["7"]
    def campo(nome): return limpar(extrair(linha, *c[nome]))

    dh_marc_raw = campo("dh_marcacao")
    dh_grav_raw = campo("dh_gravacao")
    coletor = campo("coletor")
    ligado = campo("online_offline")
    dh_marc = parse_data_hora(dh_marc_raw)
    dh_grav = parse_data_hora(dh_grav_raw)

    return {
        "tipo": "7",
        "nsr": campo("nsr"),
        "dh_marcacao_raw": dh_marc_raw,
        "dh_marcacao": dh_marc,
        "cpf": somente_digitos(campo("cpf")),
        "dh_gravacao_raw": dh_grav_raw,
        "dh_gravacao": dh_grav,
        "coletor": coletor,
        "coletor_desc": COLETOR.get(coletor, coletor),
        "online_offline": ligado,
        "modo": "off-line" if ligado == "1" else "on-line",
        "atraso_gravacao_seg": (dh_grav - dh_marc).total_seconds() if dh_marc and dh_grav else None,
        "hash": campo("hash"),
        "hash_ok": None,        # preenchido na validação da cadeia
        "hash_calculado": None,
    }

In [13]:
# =====================================================================
# CÉLULA 13 — Parser tipo "9" (trailer) e assinatura digital
# O trailer é o único registro que NÃO traz o tipo na posição 10:
# "999999999" ocupa 1-9 e o dígito "9" fica na posição 64.
# =====================================================================
def parse_tipo_9(linha: str) -> dict:
    reg = {"tipo": "9", "marcador": limpar(extrair(linha, 1, 9))}

    for tipo, (posicao, tamanho) in QTD_TRAILER.items():
        texto = somente_digitos(extrair(linha, posicao, tamanho))
        reg[f"qtd_tipo_{tipo}"] = int(texto) if texto else 0

    fechamento = limpar(extrair(linha, 64, 1))
    reg["fechamento"] = fechamento
    reg["fechamento_ok"] = fechamento == "9"
    return reg


def parse_assinatura(linha: str) -> dict:
    texto = limpar(linha)
    return {
        "tipo": "ASSINATURA",
        "conteudo": texto,
        "p7s_padrao_ok": texto == ASSINATURA_P7S,
        "tamanho": len(linha),
    }

In [14]:
# =====================================================================
# CÉLULA 14 — Dispatcher de registro e leitura do arquivo
# O trailer é detectado pelo marcador "999999999", não pela posição 10.
# =====================================================================
PARSERS = {
    "1": parse_tipo_1,
    "2": parse_tipo_2,
    "3": parse_tipo_3,
    "4": parse_tipo_4,
    "5": parse_tipo_5,
    "6": parse_tipo_6,
    "7": parse_tipo_7,
    "9": parse_tipo_9,
}


def parse_linha(linha: str, numero_linha: int | None = None) -> dict:
    linha = linha.rstrip("\r\n")

    if not linha.strip():
        return {"tipo": "BRANCO", "linha_arquivo": numero_linha}

    # Trailer: identificado pelo marcador, pois o tipo não está na posição 10
    if limpar(linha[:9]) == "999999999":
        reg = parse_tipo_9(linha)

    elif limpar(linha).startswith(ASSINATURA_P7S[:20]):
        reg = parse_assinatura(linha)

    elif len(linha) < 10:
        return {"tipo": "INVALIDO", "linha_arquivo": numero_linha,
                "erro": f"linha com apenas {len(linha)} caracteres", "bruto": linha}

    else:
        tipo = linha[9]
        parser = PARSERS.get(tipo)
        if parser is None:
            return {"tipo": tipo, "linha_arquivo": numero_linha,
                    "erro": f"tipo de registro {tipo!r} não previsto no Anexo V",
                    "bruto": linha}
        try:
            reg = parser(linha)
        except Exception as exc:
            return {"tipo": tipo, "linha_arquivo": numero_linha,
                    "erro": f"{type(exc).__name__}: {exc}", "bruto": linha}

    reg["linha_arquivo"] = numero_linha
    reg["tamanho_linha"] = len(linha)
    reg["bruto"] = linha
    return reg


def ler_afd(caminho: Path | str) -> dict:
    caminho = Path(caminho)
    with caminho.open("r", encoding=ENCODING_AFD, errors="replace", newline="") as f:
        texto = f.read()

    linhas = texto.splitlines()
    registros = [parse_linha(linha, i)
                 for i, linha in enumerate(linhas, start=1)]
    registros = [r for r in registros if r["tipo"] != "BRANCO"]

    return {
        "arquivo": caminho,
        "total_linhas": len(linhas),
        "registros": registros,
    }

In [15]:
# =====================================================================
# CÉLULA 15 — Validações do arquivo
# =====================================================================

# Fórmula da cadeia SHA-256 do tipo 7.
# Começa na variação descrita no Anexo V. A Célula 20 testa outras e a
# Célula 21 atualiza este dicionário caso o REP use algo diferente.
FORMULA_HASH = {
    "ordem": "campos+anterior",
    "cpf": "campo_12",
    "hash_anterior": "como no campo",
    "separador_nome": "nenhum",
}


def _base_hash(reg: dict, anterior: str, *, ordem: str, cpf_modo: str,
               fmt_anterior: str, sep: str) -> str:
    """Monta a base textual do SHA-256 conforme a variação escolhida."""
    bruto = reg["bruto"]

    cpf = extrair(bruto, 35, 12)
    if cpf_modo == "11 dígitos":
        cpf = somente_digitos(cpf)[-11:]

    if fmt_anterior == "minúsculo":
        ant = (anterior or "").lower()
    elif fmt_anterior == "maiúsculo":
        ant = (anterior or "").upper()
    else:
        ant = anterior or ""

    campos = sep.join([
        extrair(bruto, 1, 9),      # NSR
        extrair(bruto, 10, 1),     # tipo "7"
        extrair(bruto, 11, 24),    # DH marcação
        cpf,                       # CPF
        extrair(bruto, 47, 24),    # DH gravação
        extrair(bruto, 71, 2),     # coletor
        extrair(bruto, 73, 1),     # on-line/off-line
    ])

    return campos + sep + ant if ordem == "campos+anterior" else ant + sep + campos


def validar_nsr(registros: list[dict]) -> list[str]:
    problemas = []
    sequencia = []
    for reg in registros:
        if reg["tipo"] in ("ASSINATURA", "INVALIDO", "BRANCO", "1"):
            continue
        digitos = somente_digitos(reg.get("nsr", ""))
        if digitos:
            sequencia.append((int(digitos), reg))

    for (n1, r1), (n2, r2) in zip(sequencia, sequencia[1:]):
        if n2 != n1 + 1:
            problemas.append(
                f"NSR fora de sequência: {n1} (linha {r1['linha_arquivo']}) "
                f"-> {n2} (linha {r2['linha_arquivo']})"
            )
    return problemas


def validar_trailer(registros: list[dict]) -> list[str]:
    trailer = next((r for r in registros if r["tipo"] == "9"), None)
    if not trailer:
        return ["Trailer (tipo 9) não encontrado."]

    problemas = []
    for tipo in ("2", "3", "4", "5", "6", "7"):
        declarado = trailer.get(f"qtd_tipo_{tipo}", 0)
        real = sum(1 for r in registros if r["tipo"] == tipo)
        if declarado != real:
            problemas.append(
                f"Trailer declara {declarado} registro(s) tipo {tipo}; arquivo contém {real}."
            )
    return problemas


def validar_crc_registros(registros: list[dict]) -> list[str]:
    problemas = []
    for reg in registros:
        if reg["tipo"] not in TIPOS_COM_CRC:
            continue
        bruto = reg.get("bruto", "")
        if len(bruto) < 10:
            continue
        if not crc_valido(bruto):
            problemas.append(
                f"CRC-16 inválido no NSR {reg.get('nsr')} tipo {reg['tipo']} "
                f"(linha {reg['linha_arquivo']})."
            )
    return problemas


def validar_hash_tipo_7(registros: list[dict],
                        formula: dict | None = None) -> list[str]:
    """
    Cadeia SHA-256 do tipo 7.

    O 1º registro NÃO é verificável quando o arquivo é um extrato por período:
    o hash dele aponta para um registro anterior que não está no arquivo.
    Por isso ele é usado apenas como SEMENTE e a verificação começa no 2º.
    """
    marcs = [r for r in registros if r["tipo"] == "7" and r.get("hash")]
    if len(marcs) < 2:
        return ["Não há registros tipo 7 suficientes para validar a cadeia de hash."]

    f = dict(FORMULA_HASH)
    if formula:
        f.update(formula)

    sep = "" if f.get("separador_nome", "nenhum") == "nenhum" else "|"

    seed = marcs[0]["hash"]
    marcs[0]["hash_calculado"] = None
    # semente: hash de registro fora do arquivo
    marcs[0]["hash_ok"] = None

    problemas = []
    for reg in marcs[1:]:
        base = _base_hash(
            reg, seed,
            ordem=f["ordem"],
            cpf_modo=f["cpf"],
            fmt_anterior=f["hash_anterior"],
            sep=sep,
        )
        calculado = hashlib.sha256(
            base.encode(ENCODING_AFD, errors="replace")
        ).hexdigest()

        reg["hash_calculado"] = calculado
        reg["hash_ok"] = calculado.lower() == (reg["hash"] or "").lower()

        if not reg["hash_ok"]:
            problemas.append(
                f"Hash divergente no NSR {reg['nsr']} (linha {reg['linha_arquivo']})."
            )

        seed = reg["hash"]

    return problemas


def validar_estrutura(registros: list[dict]) -> list[str]:
    problemas = []
    tipos = [r["tipo"] for r in registros]

    if not tipos or tipos[0] != "1":
        problemas.append(
            "O arquivo não começa com registro tipo 1 (cabeçalho).")

    if "9" not in tipos:
        problemas.append("O arquivo não contém registro tipo 9 (trailer).")
    else:
        posteriores = [t for t in tipos[tipos.index(
            "9") + 1:] if t != "ASSINATURA"]
        if posteriores:
            problemas.append(
                f"Existem registros após o trailer: {sorted(set(posteriores))}.")

    for reg in registros:
        if "erro" in reg:
            problemas.append(f"Linha {reg['linha_arquivo']}: {reg['erro']}")

    return problemas

In [16]:
# =====================================================================
# CÉLULA 16 — Consolidação em DataFrames
# =====================================================================
def filtrar_tipo(registros: list[dict], tipo: str) -> list[dict]:
    return [r for r in registros if r["tipo"] == tipo]


def dataframe_tipo(registros: list[dict], tipo: str) -> pd.DataFrame:
    linhas = filtrar_tipo(registros, tipo)
    if not linhas:
        return pd.DataFrame()
    df = pd.DataFrame(linhas)

    if "cpf" in df.columns:
        df["cpf_formatado"] = df["cpf"].apply(formatar_cpf)

    if tipo == "7" and "dh_marcacao" in df.columns:
        df = df.sort_values(["cpf", "dh_marcacao"],
                            na_position="last").reset_index(drop=True)
        df["data"] = df["dh_marcacao"].dt.date
        df["hora"] = df["dh_marcacao"].dt.strftime("%H:%M:%S")

    return df


def montar_dataframes(registros: list[dict]) -> dict[str, pd.DataFrame]:
    return {
        "cabecalho": dataframe_tipo(registros, "1"),
        "empresa_rep": dataframe_tipo(registros, "2"),
        "marcacoes_repca": dataframe_tipo(registros, "3"),
        "ajustes_relogio": dataframe_tipo(registros, "4"),
        "empregados": dataframe_tipo(registros, "5"),
        "eventos": dataframe_tipo(registros, "6"),
        "marcacoes_repp": dataframe_tipo(registros, "7"),
        "trailer": dataframe_tipo(registros, "9"),
        "assinatura": dataframe_tipo(registros, "ASSINATURA"),
    }

In [17]:
# =====================================================================
# CÉLULA 17 — Resumo do arquivo
# =====================================================================
def resumo(resultado: dict) -> dict:
    registros = resultado["registros"]
    df_cab = dataframe_tipo(registros, "1")
    marcacoes = filtrar_tipo(registros, "7") or filtrar_tipo(registros, "3")

    cab = df_cab.iloc[0].to_dict() if not df_cab.empty else {}
    cpfs = sorted({r["cpf"] for r in marcacoes if r.get("cpf")})

    return {
        "arquivo": resultado["arquivo"].name,
        "empregador": cab.get("empregador_razao_social"),
        "documento_empregador": cab.get("empregador_documento"),
        "versao_leiaute": cab.get("versao_leiaute"),
        "periodo": f"{cab.get('data_inicial')} a {cab.get('data_final')}",
        "gerado_em": cab.get("dh_geracao"),
        "total_linhas": resultado["total_linhas"],
        "total_registros": len(registros),
        "marcacoes": len(marcacoes),
        "empregados_com_marcacao": len(cpfs),
        "contagem_por_tipo": pd.Series([r["tipo"] for r in registros]).value_counts().to_dict(),
    }


def problemas_estruturais(registros: list[dict]) -> list[str]:
    return (
        validar_estrutura(registros)
        + validar_nsr(registros)
        + validar_trailer(registros)
        + validar_crc_registros(registros)
    )


def imprimir_resumo(resultado: dict) -> None:
    info = resumo(resultado)
    print("=" * 70)
    print("RESUMO DO ARQUIVO AFD")
    print("=" * 70)
    for chave, valor in info.items():
        print(f"{chave:26}: {valor}")

    registros = resultado["registros"]

    # 1) Estrutura, NSR, trailer e CRC-16
    problemas = problemas_estruturais(registros)
    print("-" * 70)
    if problemas:
        print(f"{len(problemas)} inconsistência(s) estrutural(is):")
        for p in problemas[:20]:
            print("  •", p)
        if len(problemas) > 20:
            print(f"  ... e mais {len(problemas) - 20}.")
    else:
        print("Estrutura, NSR, trailer e CRC-16: OK")

    # 2) Cadeia de hash do tipo 7 (relatada à parte, pois depende da fórmula)
    marcs = [r for r in registros if r["tipo"] == "7"]
    if marcs:
        divergencias = validar_hash_tipo_7(registros)
        verificados = sum(1 for r in marcs if r.get("hash_ok") is not None)
        aprovados = sum(1 for r in marcs if r.get("hash_ok") is True)

        print("-" * 70)
        print(f"Cadeia SHA-256 (tipo 7): {aprovados}/{verificados} verificados "
              f"({len(marcs)} marcações, 1ª é semente)")
        for p in divergencias[:3]:
            print("  •", p)
        if len(divergencias) > 3:
            print(f"  ... e mais {len(divergencias) - 3}. Rode a Célula 20 para "
                  f"descobrir a fórmula usada pelo REP.")

    print("=" * 70)

In [18]:
# =====================================================================
# CÉLULA 18 — Exportação dos resultados
# O Excel não aceita datetime com timezone nem dict na célula.
# =====================================================================
def normalizar_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """Remove timezone e converte tipos que o openpyxl não serializa."""
    df = df.copy()
    for col in df.columns:
        serie = df[col]
        if isinstance(serie.dtype, pd.DatetimeTZDtype):
            df[col] = serie.dt.tz_localize(None)
        elif serie.dtype == object:
            df[col] = serie.map(
                lambda v: v.replace(tzinfo=None)
                if isinstance(v, datetime) and v.tzinfo
                else (str(v) if isinstance(v, (dict, list, set)) else v)
            )
    return df


def exportar_excel(resultado: dict, destino: Path | str | None = None) -> Path:
    destino = Path(destino or (
        resultado["arquivo"].parent /
        f"{resultado['arquivo'].stem}_analisado.xlsx"
    ))
    dfs = montar_dataframes(resultado["registros"])

    resumo_df = pd.DataFrame([resumo(resultado)]).T.reset_index()
    resumo_df.columns = ["campo", "valor"]
    resumo_df = normalizar_para_excel(resumo_df)

    with pd.ExcelWriter(destino, engine="openpyxl",
                        datetime_format="dd/mm/yyyy hh:mm:ss") as writer:
        resumo_df.to_excel(writer, sheet_name="resumo", index=False)

        for nome, df in dfs.items():
            if not df.empty:
                normalizar_para_excel(df).to_excel(
                    writer, sheet_name=nome[:31], index=False
                )

    return destino


def exportar_csvs(resultado: dict, pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or resultado["arquivo"].parent)
    gerados = []
    for nome, df in montar_dataframes(resultado["registros"]).items():
        if df.empty:
            continue
        caminho = pasta / f"{resultado['arquivo'].stem}_{nome}.csv"
        normalizar_para_excel(df).to_csv(
            caminho, index=False, sep=";", encoding="utf-8-sig")
        gerados.append(caminho)
    return gerados

In [19]:
# =====================================================================
# CÉLULA 19 — Pipeline: lê todos os arquivos da pasta e exporta
# =====================================================================
def processar_arquivo(caminho: Path, exportar: bool = True) -> dict:
    resultado = ler_afd(caminho)
    imprimir_resumo(resultado)

    dfs = montar_dataframes(resultado["registros"])
    resultado["dataframes"] = dfs

    if not dfs["marcacoes_repp"].empty:
        print("\nAmostra das marcações REP-P:")
        print(dfs["marcacoes_repp"][["nsr", "cpf_formatado",
              "dh_marcacao", "modo", "coletor_desc"]].head())
    elif not dfs["marcacoes_repca"].empty:
        print("\nAmostra das marcações REP-C/REP-A:")
        print(dfs["marcacoes_repca"]
              [["nsr", "cpf_formatado", "dh_marcacao"]].head())

    if exportar:
        print("\nPlanilha gerada:", exportar_excel(resultado).name)

    return resultado


RESULTADOS = []
for arquivo in ARQUIVOS:
    print()
    RESULTADOS.append(processar_arquivo(arquivo))

# Acesso rápido ao último arquivo processado
if RESULTADOS:
    ultimo = RESULTADOS[-1]
    df_cabecalho = ultimo["dataframes"]["cabecalho"]
    df_marcacoes = ultimo["dataframes"]["marcacoes_repp"]


RESUMO DO ARQUIVO AFD
arquivo                   : AFDBR512021002936436585411000133REP_P.txt
empregador                : TOKUNAGA COMERCIO DE RACOES LTDA
documento_empregador      : 36.585.411/0001-33
versao_leiaute            : 003
periodo                   : 2026-09-26 a 2026-10-25
gerado_em                 : 2026-10-26 12:40:00-03:00
total_linhas              : 837
total_registros           : 837
marcacoes                 : 834
empregados_com_marcacao   : 11
contagem_por_tipo         : {'7': 834, '1': 1, '9': 1, 'ASSINATURA': 1}
----------------------------------------------------------------------
Estrutura, NSR, trailer e CRC-16: OK
----------------------------------------------------------------------
Cadeia SHA-256 (tipo 7): 833/833 verificados (834 marcações, 1ª é semente)

Amostra das marcações REP-P:
         nsr   cpf_formatado               dh_marcacao     modo       coletor_desc
0  000049870  092.819.288-17 2026-09-26 07:16:00-03:00  on-line  aplicativo mobile
1  000049880

In [20]:
# =====================================================================
# CÉLULA 20 — Diagnóstico da fórmula do hash do tipo 7
# Verifica a cadeia a partir do 2º registro, usando o 1º como semente.
# =====================================================================
def diagnosticar_hash(registros: list[dict], limite: int = 8) -> pd.DataFrame:
    marcs = [r for r in registros if r["tipo"] == "7" and r.get("hash")]
    if len(marcs) < 3:
        return pd.DataFrame()

    testes = []
    janela = marcs[1:limite + 1]

    for ordem in ("campos+anterior", "anterior+campos"):
        for cpf_modo in ("campo_12", "11 dígitos"):
            for fmt_anterior in ("como no campo", "minúsculo", "maiúsculo"):
                for sep_nome, sep in (("nenhum", ""), ("pipe", "|")):
                    seed = marcs[0]["hash"]
                    verificados, falhou = 0, None

                    for reg in janela:
                        base = _base_hash(
                            reg, seed,
                            ordem=ordem, cpf_modo=cpf_modo,
                            fmt_anterior=fmt_anterior, sep=sep,
                        )
                        calc = hashlib.sha256(
                            base.encode(ENCODING_AFD, errors="replace")
                        ).hexdigest()

                        if calc.lower() != (reg["hash"] or "").lower():
                            falhou = reg["nsr"]
                            break

                        verificados += 1
                        seed = reg["hash"]

                    testes.append({
                        "ordem": ordem,
                        "cpf": cpf_modo,
                        "hash_anterior": fmt_anterior,
                        "separador_nome": sep_nome,
                        "verificados": verificados,
                        "cadeia_ok": falhou is None,
                        "falhou_no_nsr": falhou or "—",
                    })

    return (
        pd.DataFrame(testes)
        .sort_values(["cadeia_ok", "verificados"], ascending=[False, False])
        .reset_index(drop=True)
    )


DIAGNOSTICO_HASH = (
    diagnosticar_hash(RESULTADOS[-1]["registros"]
                      ) if RESULTADOS else pd.DataFrame()
)
DIAGNOSTICO_HASH.head(10)

,ordem,cpf,hash_anterior,separador_nome,verificados,cadeia_ok,falhou_no_nsr
0,campos+anterior,campo_12,como no campo,nenhum,8,True,—
1,campos+anterior,campo_12,minúsculo,nenhum,8,True,—
2,campos+anterior,campo_12,como no campo,pipe,0,False,000049867
3,campos+anterior,campo_12,minúsculo,pipe,0,False,000049867
4,campos+anterior,campo_12,maiúsculo,nenhum,0,False,000049867
5,campos+anterior,campo_12,maiúsculo,pipe,0,False,000049867
6,campos+anterior,11 dígitos,como no campo,nenhum,0,False,000049867
7,campos+anterior,11 dígitos,como no campo,pipe,0,False,000049867
8,campos+anterior,11 dígitos,minúsculo,nenhum,0,False,000049867
9,campos+anterior,11 dígitos,minúsculo,pipe,0,False,000049867


In [21]:
# =====================================================================
# CÉLULA 21 — Aplicar a fórmula detectada e revalidar o arquivo
# =====================================================================
CHAVES_FORMULA = ("ordem", "cpf", "hash_anterior", "separador_nome")


def formula_vencedora(df_testes: pd.DataFrame) -> dict | None:
    if df_testes is None or df_testes.empty:
        return None
    ok = df_testes[df_testes["cadeia_ok"]]
    if ok.empty:
        return None
    linha = ok.iloc[0]
    return {chave: linha[chave] for chave in CHAVES_FORMULA}


VENCEDORA = formula_vencedora(DIAGNOSTICO_HASH)

if VENCEDORA:
    FORMULA_HASH.clear()
    FORMULA_HASH.update(VENCEDORA)
    print("Fórmula detectada:", VENCEDORA)
else:
    print("Nenhuma das combinações fechou — rode a Célula 22 e envie a saída.")

print()

if RESULTADOS:
    registros = RESULTADOS[-1]["registros"]
    divergencias = validar_hash_tipo_7(registros)
    marcs = [r for r in registros if r["tipo"] == "7"]
    verificados = sum(1 for r in marcs if r.get("hash_ok") is not None)
    aprovados = sum(1 for r in marcs if r.get("hash_ok") is True)

    print("=" * 70)
    print("CADEIA SHA-256 — REGISTROS TIPO 7")
    print("=" * 70)
    print(f"fórmula aplicada : {FORMULA_HASH}")
    print(f"registros tipo 7 : {len(marcs)}")
    print(
        f"semente          : {marcs[0]['nsr'] if marcs else '—'} (não verificável)")
    print(f"verificados      : {verificados}")
    print(f"aprovados        : {aprovados}")

    if divergencias:
        print(f"\n{len(divergencias)} divergência(s):")
        for p in divergencias[:10]:
            print("  •", p)
        if len(divergencias) > 10:
            print(f"  ... e mais {len(divergencias) - 10}.")
    else:
        print("\nCadeia íntegra do 2º ao último registro.")
    print("=" * 70)

Fórmula detectada: {'ordem': 'campos+anterior', 'cpf': 'campo_12', 'hash_anterior': 'como no campo', 'separador_nome': 'nenhum'}

CADEIA SHA-256 — REGISTROS TIPO 7
fórmula aplicada : {'ordem': 'campos+anterior', 'cpf': 'campo_12', 'hash_anterior': 'como no campo', 'separador_nome': 'nenhum'}
registros tipo 7 : 834
semente          : 000049866 (não verificável)
verificados      : 833
aprovados        : 833

Cadeia íntegra do 2º ao último registro.


In [22]:
# =====================================================================
# CÉLULA 22 — Despejo dos campos usados no hash (mascarado por padrão)
# Use mostrar_dados=True apenas localmente: o dado bruto contém CPF.
# =====================================================================
def mascarar_cpf(valor: str) -> str:
    """Mostra só o início e o fim do CPF."""
    d = somente_digitos(valor)[-11:]
    return f"{d[:3]}.***.***-{d[9:]}" if len(d) == 11 else "***"


def mascarar_hash(valor: str) -> str:
    """Mostra só os 12 primeiros caracteres do SHA-256."""
    valor = valor or ""
    return f"{valor[:12]}…{len(valor)} chars"


def despejar_hash(registros: list[dict], n: int = 2,
                  mostrar_dados: bool = False) -> None:
    marcs = [r for r in registros if r["tipo"]
             == "7" and r.get("hash")][:n + 1]
    for reg in marcs:
        b = reg["bruto"]
        cpf = extrair(b, 35, 12)
        h = reg["hash"]
        print(f"NSR      : {extrair(b, 1, 9)}")
        print(f"DH marc. : {extrair(b, 11, 24)}")
        print(f"CPF      : {cpf if mostrar_dados else mascarar_cpf(cpf)}")
        print(f"DH grav. : {extrair(b, 47, 24)}")
        print(
            f"coletor  : '{extrair(b, 71, 2)}'  on/off: '{extrair(b, 73, 1)}'")
        print(f"hash     : {h if mostrar_dados else mascarar_hash(h)}")
        print(f"len linha: {len(b)}")
        print("-" * 70)


if RESULTADOS:
    despejar_hash(RESULTADOS[-1]["registros"], mostrar_dados=False)

NSR      : 000049866
DH marc. : 2026-09-26T06:19:00-0300
CPF      : 299.***.***-22
DH grav. : 2026-09-26T06:19:00-0300
coletor  : '01'  on/off: '0'
hash     : 22196c735021…64 chars
len linha: 137
----------------------------------------------------------------------
NSR      : 000049867
DH marc. : 2026-09-26T06:33:00-0300
CPF      : 375.***.***-09
DH grav. : 2026-09-26T06:33:00-0300
coletor  : '01'  on/off: '0'
hash     : 014ea278fb1e…64 chars
len linha: 137
----------------------------------------------------------------------
NSR      : 000049868
DH marc. : 2026-09-26T06:59:00-0300
CPF      : 308.***.***-00
DH grav. : 2026-09-26T06:59:00-0300
coletor  : '01'  on/off: '0'
hash     : 0dd3d40e6c84…64 chars
len linha: 137
----------------------------------------------------------------------


In [23]:
# =====================================================================
# CÉLULA 23 — Marcações por dia e pareamento entrada/saída
# =====================================================================
CUTOFF_MADRUGADA = 5      # marcas antes das 5h pertencem à jornada anterior
TOLERANCIA_DUPLICIDADE = 60  # segundos
MAX_PARES = 4


def preparar_marcacoes(registros: list[dict],
                       cutoff: int = CUTOFF_MADRUGADA) -> pd.DataFrame:
    """Uma linha por marcação, com dia de jornada já ajustado."""
    linhas = []
    for r in registros:
        if r["tipo"] not in ("3", "7"):
            continue
        dh = r.get("dh_marcacao")
        if dh is None:
            continue
        linhas.append({
            "nsr": r.get("nsr"),
            "cpf": r.get("cpf"),
            "dh": dh,
            "modo": r.get("modo", ""),
            "coletor": r.get("coletor_desc", ""),
            "atraso_gravacao_seg": r.get("atraso_gravacao_seg"),
        })

    df = pd.DataFrame(linhas)
    if df.empty:
        return df

    df["cpf_formatado"] = df["cpf"].apply(formatar_cpf)

    # Dia de jornada: marca antes do cutoff pertence ao dia anterior
    hora_frac = df["dh"].dt.hour + df["dh"].dt.minute / 60
    data = pd.to_datetime(df["dh"].dt.date)
    df["data_jornada"] = (
        data - pd.to_timedelta((hora_frac < cutoff).astype(int), unit="D")
    ).dt.date

    return df.sort_values(["cpf", "dh"]).reset_index(drop=True)


def remover_duplicidades(df: pd.DataFrame,
                         tolerancia: int = TOLERANCIA_DUPLICIDADE):
    """Remove marcas repetidas da mesma pessoa dentro da tolerância."""
    if df.empty:
        return df, df

    df = df.sort_values(["cpf", "dh"]).copy()
    df["delta_seg"] = df.groupby("cpf")["dh"].diff().dt.total_seconds()

    duplicada = df["delta_seg"].notna() & (df["delta_seg"] < tolerancia)
    descartadas = df[duplicada].copy()
    limpo = df[~duplicada].drop(columns="delta_seg").reset_index(drop=True)

    return limpo, descartadas


def parear_por_dia(df: pd.DataFrame, max_pares: int = MAX_PARES) -> pd.DataFrame:
    """Converte as marcas do dia em entrada_N / saida_N."""
    if df.empty:
        return pd.DataFrame()

    linhas = []
    for (cpf, dia), grupo in df.groupby(["cpf", "data_jornada"], sort=True):
        grupo = grupo.sort_values("dh")
        dhs = grupo["dh"].tolist()

        reg = {
            "cpf": cpf,
            "cpf_formatado": grupo["cpf_formatado"].iloc[0],
            "data": dia,
            "qtd_marcacoes": len(dhs),
            "nsrs": ", ".join(str(x) for x in grupo["nsr"]),
        }

        for n in range(1, max_pares + 1):
            i = (n - 1) * 2
            entrada = dhs[i] if i < len(dhs) else None
            saida = dhs[i + 1] if i + 1 < len(dhs) else None
            reg[f"entrada_{n}"] = entrada
            reg[f"saida_{n}"] = saida
            reg[f"horas_{n}"] = (
                (saida - entrada).total_seconds() / 3600
                if entrada and saida else None
            )

        total = sum(v for k, v in reg.items()
                    if k.startswith("horas_") and v)
        reg["horas_total"] = total if total else None

        pendencias = []
        if len(dhs) == 1:
            pendencias.append("marcação única")
        elif len(dhs) % 2:
            pendencias.append("marcação sem par")
        if len(dhs) < 4 and len(dhs) > 1:
            pendencias.append("menos de 4 marcações")
        if len(dhs) > max_pares * 2:
            pendencias.append(f"{len(dhs)} marcações — revisar")
        reg["situacao"] = "; ".join(pendencias) if pendencias else "ok"

        linhas.append(reg)

    return (
        pd.DataFrame(linhas)
        .sort_values(["cpf", "data"])
        .reset_index(drop=True)
    )


# --- execução ---------------------------------------------------------
if RESULTADOS:
    df_marc = preparar_marcacoes(RESULTADOS[-1]["registros"])
    df_marc, df_duplicadas = remover_duplicidades(df_marc)
    DF_DIAS = parear_por_dia(df_marc)

    print(f"marcações válidas     : {len(df_marc)}")
    print(f"duplicidades removidas: {len(df_duplicadas)}")
    print(f"dias de jornada       : {len(DF_DIAS)}")
    print(f"dias com pendência    : {(DF_DIAS['situacao'] != 'ok').sum()}")

    print("\nAmostra (2 primeiros dias de 2 empregados):")
    amostra = (DF_DIAS.groupby("cpf_formatado").head(2).head(6)
               if not DF_DIAS.empty else DF_DIAS)
    print(amostra[["cpf_formatado", "data", "entrada_1", "saida_1",
                   "entrada_2", "saida_2", "horas_total", "situacao"]])

marcações válidas     : 818
duplicidades removidas: 16
dias de jornada       : 222
dias com pendência    : 106

Amostra (2 primeiros dias de 2 empregados):
     cpf_formatado        data                 entrada_1                   saida_1                 entrada_2                   saida_2  horas_total                                situacao
0   092.819.288-17  2026-09-26 2026-09-26 07:16:00-03:00 2026-09-26 11:44:00-03:00 2026-09-26 13:10:00-03:00                       NaT     4.466667  marcação sem par; menos de 4 marcações
1   092.819.288-17  2026-09-28 2026-09-28 07:31:00-03:00 2026-09-28 12:01:00-03:00 2026-09-28 12:02:00-03:00 2026-09-28 13:30:00-03:00     5.983333                        marcação sem par
24  141.906.768-02  2026-09-26 2026-09-26 07:00:00-03:00 2026-09-26 09:13:00-03:00 2026-09-26 09:41:00-03:00 2026-09-26 12:20:00-03:00     4.866667                                      ok
25  141.906.768-02  2026-09-28 2026-09-28 06:58:00-03:00 2026-09-28 09:22:00-03:00 2026-09-2

In [24]:
# =====================================================================
# CÉLULA 24 — Consolidado por empregado
# =====================================================================
def consolidar_por_empregado(df_dias: pd.DataFrame) -> pd.DataFrame:
    if df_dias.empty:
        return pd.DataFrame()

    resumo = df_dias.groupby("cpf_formatado").agg(
        dias_com_marcacao=("data", "nunique"),
        dias_com_pendencia=("situacao", lambda s: (s != "ok").sum()),
        total_horas=("horas_total", "sum"),
        media_horas_dia=("horas_total", "mean"),
    ).reset_index()

    return resumo.sort_values("total_horas", ascending=False)


if RESULTADOS and not DF_DIAS.empty:
    DF_CONSOLIDADO = consolidar_por_empregado(DF_DIAS)
    print(DF_CONSOLIDADO.to_string(index=False))

 cpf_formatado  dias_com_marcacao  dias_com_pendencia  total_horas  media_horas_dia
397.205.418-14                 23                   3   193.833333         8.427536
308.951.818-00                 24                   6   180.166667         7.506944
092.819.288-17                 24                   9   175.650000         7.318750
375.468.798-09                 24                  20   175.383333         8.769167
536.779.468-83                 22                  12   171.283333         7.785606
716.314.364-81                 22                   7   157.883333         7.176515
141.906.768-02                 22                   6   157.500000         7.500000
299.325.948-22                 21                  21   131.583333        13.158333
334.788.598-83                 24                  19   122.466667         5.566667
557.053.428-88                 12                   1   101.050000         8.420833
405.139.758-02                  4                   2    24.266667         6

In [25]:
# =====================================================================
# CÉLULA 25 — Cadastro de colaboradores (nome por CPF)
# Lê Colaboradores.xlsx da raiz do projeto. Caminho dinâmico:
#   1. variável de ambiente AFD_COLABORADORES
#   2. <BASE_DIR>/Colaboradores.xlsx   (notebook, script e .exe)
#   3. <_MEIPASS>/Colaboradores.xlsx   (PyInstaller --onefile)
# =====================================================================
ARQUIVO_COLABORADORES = "Colaboradores.xlsx"


def resolver_arquivo_colaboradores(nome: str = ARQUIVO_COLABORADORES) -> Path:
    env = os.getenv("AFD_COLABORADORES")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]

    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_file():
            return candidato

    return BASE_DIR / nome


def normalizar_cpf(valor) -> str:
    """Aceita número, float ou texto e devolve sempre 11 dígitos."""
    if valor is None:
        return ""
    if isinstance(valor, float):
        if pd.isna(valor):
            return ""
        valor = int(valor)

    d = somente_digitos(str(valor))

    # Campo de 12 posições do AFD: o zero à esquerda é preenchimento
    if len(d) == 12 and d.startswith("0"):
        d = d[1:]

    return d.zfill(11) if 0 < len(d) <= 11 else d[-11:]


def cpf_valido(cpf: str) -> bool:
    """Confere os dois dígitos verificadores do CPF."""
    d = normalizar_cpf(cpf)
    if len(d) != 11 or len(set(d)) == 1:
        return False

    for tamanho in (9, 10):
        soma = sum(int(d[i]) * (tamanho + 1 - i) for i in range(tamanho))
        digito = 11 - soma % 11
        digito = 0 if digito >= 10 else digito
        if digito != int(d[tamanho]):
            return False

    return True


def _normalizar_coluna(nome: str) -> str:
    """minúsculo e sem acento, para localizar colunas pelo nome."""
    tabela = str.maketrans(
        "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc"
    )
    return str(nome).lower().translate(tabela)


def carregar_colaboradores(caminho: Path | str | None = None) -> pd.DataFrame:
    caminho = Path(caminho or resolver_arquivo_colaboradores())
    vazio = pd.DataFrame(columns=["cpf", "nome"])

    if not caminho.is_file():
        print(f"[colaboradores] arquivo não encontrado: {caminho}")
        return vazio

    # A planilha pode ter aba de contador, de resumo etc.: escolhe a que
    # realmente tem coluna de CPF e coluna de nome.
    abas = pd.read_excel(caminho, sheet_name=None, dtype=str)

    escolhida = None
    for nome_aba, df_aba in abas.items():
        colunas = [_normalizar_coluna(c) for c in df_aba.columns]
        tem_cpf = any("cpf" in c for c in colunas)
        tem_nome = any("nome" in c or "colaborador" in c for c in colunas)
        if tem_cpf and tem_nome:
            escolhida = (nome_aba, df_aba)
            break

    if escolhida is None:
        print(f"[colaboradores] nenhuma aba com nome e CPF em {caminho.name}")
        print(f"[colaboradores] abas encontradas: {list(abas)}")
        return vazio

    nome_aba, df = escolhida

    col_cpf = next(c for c in df.columns if "cpf" in _normalizar_coluna(c))
    col_nome = next(
        c for c in df.columns
        if "nome" in _normalizar_coluna(c)
        or "colaborador" in _normalizar_coluna(c)
    )

    base = pd.DataFrame({
        "cpf": df[col_cpf].map(normalizar_cpf),
        "nome": df[col_nome].fillna("").astype(str).str.strip(),
    })
    base = base[(base["cpf"].str.len() == 11) & (base["nome"] != "")]

    validos = base[base["cpf"].map(cpf_valido)].copy()
    invalidos = base[~base["cpf"].map(cpf_valido)]

    for _, linha in invalidos.iterrows():
        print(f"[colaboradores] CPF inválido ignorado: "
              f"{mascarar_cpf(linha['cpf'])} — {linha['nome']}")

    # Um CPF repetido com nomes diferentes: mantém o primeiro e avisa
    duplicados = validos[validos.duplicated("cpf", keep=False)]
    if not duplicados.empty:
        print(f"[colaboradores] CPF repetido no cadastro: "
              f"{duplicados['cpf'].nunique()} caso(s) — mantido o primeiro nome")

    validos = (
        validos.drop_duplicates("cpf", keep="first")
        .sort_values("nome")
        .reset_index(drop=True)
    )

    print(f"[colaboradores] {len(validos)} registro(s) da aba '{nome_aba}' "
          f"em {caminho.name}")
    return validos[["cpf", "nome"]]


def mapear_nomes(df: pd.DataFrame,
                 colaboradores: pd.DataFrame | None = None) -> pd.DataFrame:
    """Adiciona a coluna 'nome' a um DataFrame que tenha a coluna 'cpf'."""
    cadastro = COLABORADORES if colaboradores is None else colaboradores
    df = df.copy()

    if df.empty:
        if "nome" not in df.columns:
            df["nome"] = pd.Series(dtype="object")
        return df

    if cadastro.empty:
        df["nome"] = "não cadastrado"
        return df

    mapa = dict(zip(cadastro["cpf"], cadastro["nome"]))
    df["nome"] = df["cpf"].map(normalizar_cpf).map(mapa).fillna("não cadastrado")
    return df


def auditar_colaboradores(df_marc: pd.DataFrame,
                          colaboradores: pd.DataFrame | None = None) -> None:
    """Compara os CPFs das marcações com o cadastro."""
    cadastro = COLABORADORES if colaboradores is None else colaboradores

    if df_marc.empty:
        print("Sem marcações para comparar.")
        return

    cpfs_afd = set(df_marc["cpf"].map(normalizar_cpf))
    cpfs_cad = set(cadastro["cpf"]) if not cadastro.empty else set()

    sem_cadastro = sorted(cpfs_afd - cpfs_cad)
    sem_marcacao = sorted(cpfs_cad - cpfs_afd)

    print("=" * 70)
    print("CADASTRO × MARCAÇÕES")
    print("=" * 70)
    print(f"CPFs com marcação     : {len(cpfs_afd)}")
    print(f"CPFs no cadastro      : {len(cpfs_cad)}")
    print(f"Casados               : {len(cpfs_afd & cpfs_cad)}")

    if sem_cadastro:
        print(f"\n{len(sem_cadastro)} CPF(s) com marcação SEM cadastro:")
        for c in sem_cadastro:
            qtd = (df_marc["cpf"].map(normalizar_cpf) == c).sum()
            print(f"  • {mascarar_cpf(c)} — {qtd} marcação(ões)")
    else:
        print("\nTodos os CPFs com marcação estão cadastrados.")

    if sem_marcacao:
        print(f"\n{len(sem_marcacao)} CPF(s) no cadastro SEM marcação no período:")
        for c in sem_marcacao:
            nome = cadastro.loc[cadastro["cpf"] == c, "nome"].iloc[0]
            print(f"  • {mascarar_cpf(c)} — {nome}")

    print("=" * 70)


# --- execução ---------------------------------------------------------
ARQ_COLABORADORES = resolver_arquivo_colaboradores()
print("Cadastro:", ARQ_COLABORADORES)
COLABORADORES = carregar_colaboradores(ARQ_COLABORADORES)

if RESULTADOS:
    DF_MARC = mapear_nomes(df_marc)          # da Célula 23
    DF_DIAS = mapear_nomes(DF_DIAS)
    auditar_colaboradores(DF_MARC)

    print("\nColunas de DF_DIAS agora:")
    print(list(DF_DIAS.columns))

Cadastro: \\AGROSRV\Automacoes\Marcacao_Pontos\Colaboradores.xlsx
[colaboradores] 11 registro(s) da aba 'Plan1' em Colaboradores.xlsx
CADASTRO × MARCAÇÕES
CPFs com marcação     : 11
CPFs no cadastro      : 11
Casados               : 11

Todos os CPFs com marcação estão cadastrados.

Colunas de DF_DIAS agora:
['cpf', 'cpf_formatado', 'data', 'qtd_marcacoes', 'nsrs', 'entrada_1', 'saida_1', 'horas_1', 'entrada_2', 'saida_2', 'horas_2', 'entrada_3', 'saida_3', 'horas_3', 'entrada_4', 'saida_4', 'horas_4', 'horas_total', 'situacao', 'nome']
